In [ ]:
근데 오늘 작성한 파일 2개 중

text_termination = TextMentionTermination("APPROVED")

max_message_termination = MaxMessageTermination(max_messages=50)

termination_condition = text_termination | max_message_termination



team = SelectorGroupChat(

    participants=[

        research_agent,

        research_analyst,

        research_enhancer,

        research_planner,

        quality_reviewer,

        user_proxy,

    ],

    selector_prompt=selector_prompt,

    model_client=model_client,

    # allow_repeated_speaker=True,

    termination_condition=termination_condition,

)

이 코드에서 selectorgroupchat 함수 내 인자로 받는 거랑 전역 변수로 설정해놓은거(텍스트 길이) 이름이 똑같잖아. 원래는 이러면 에러 나는거 아니야? 근데 보통 보면 이런 규칙으로 작성하기는 하는데 가독성, 직관성은 알겠는데 규칙에 혼란이 오네

In [2]:
import os
from dotenv import load_dotenv

# 현재 폴더에 있는 .env 파일을 읽어서 환경 변수로 등록
load_dotenv()

True

In [3]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import web_search_tool, save_report_to_md

In [4]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
)

In [5]:
# 해당 agent는 selector agent(각 agent가 어떤 작업을 할지 할당해주는 애)가 설명을 참고해서 작업을 수행할 수 있게
# agent가 뭘하는지에 따라 selector ai가 누가 말할지 정해줌
# description 작성해야 함. autogen 그냥 사용할 때는 system_message만 작성함. 지금은 deep research를 위해 수행

# research plan 만들고, 우리가 만들어야 할 쿼리 생각하기
# 뭐 어떤 분야에서 검색할지 등
research_planner = AssistantAgent(
    "research_planner",
    description="A strategic research coordinator that breaks down complex questions into research subtasks",
    model_client=model_client,
    system_message="""You are a research planning specialist. Your job is to create a focused research plan.

For each research question, create a FOCUSED research plan with:

1. **Core Topics**: 2-3 main areas to investigate
2. **Search Queries**: Create 3-5 specific search queries covering:
   - Latest developments and news
   - Key statistics or data
   - Expert analysis or studies
   - Future outlook

Keep the plan focused and achievable. Quality over quantity.""",
)

# plan에서 나온 찾을 거 websearchtool로 검색해서 정보 모음
research_agent = AssistantAgent(
    "research_agent",
    description="A web research specialist that searches and extracts content",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""You are a web research specialist. Your job is to conduct focused searches based on the research plan.

RESEARCH STRATEGY:
1. **Execute 3-5 searches** from the research plan
2. **Extract key information** from the results:
   - Main facts and statistics
   - Recent developments
   - Expert opinions
   - Important context

3. **Quality focus**:
   - Prioritize authoritative sources
   - Look for recent information (within 2 years)
   - Note diverse perspectives

After completing the searches from the plan, summarize what you found. Your goal is to gather 5-10 quality sources.""",
)

# 긁어모은 정보로 보고서 만들기. 양식은 아래 create
research_analyst = AssistantAgent(
    "research_analyst",
    description="An expert analyst that creates research reports",
    model_client=model_client,
    system_message="""You are a research analyst. Create a comprehensive report from the gathered research.

CREATE A RESEARCH REPORT with:

## Executive Summary
- Key findings and conclusions
- Main insights

## Background & Current State
- Current landscape
- Recent developments
- Key statistics and data

## Analysis & Insights
- Main trends
- Different perspectives
- Expert opinions

## Future Outlook
- Emerging trends
- Predictions
- Implications

## Sources
- List all sources used

Write a clear, well-structured report based on the research gathered. End with "REPORT_COMPLETE" when finished.""",
)

# report 살펴보는 애
quality_reviewer = AssistantAgent(
    "quality_reviewer",
    description="A quality assurance specialist that evaluates research completeness and accuracy",
    tools=[save_report_to_md],
    model_client=model_client,
    system_message="""You are a quality reviewer. Your job is to check if the research analyst has produced a complete research report.

Look for:
- A comprehensive research report from the research analyst that ends with "REPORT_COMPLETE"
- The research question is fully answered
- Sources are cited and reliable
- The report includes summary, key information, analysis, and sources

When you see a complete research report that ends with "REPORT_COMPLETE":
1. First, use the save_report_to_md tool to save the report to report.md
2. Then say: "The research is complete. The report has been saved to report.md. Please review the report and let me know if you approve it or need additional research."

If the research analyst has NOT yet created a complete report, tell them to create one now.""",
)

# 빈 곳이 있나 확인하는 작업 수행. 조사에서 빠진 게 있는지 체크
research_enhancer = AssistantAgent(
    "research_enhancer",
    description="A specialist that identifies critical gaps only",
    model_client=model_client,
    system_message="""You are a research enhancement specialist. Your job is to identify ONLY CRITICAL gaps.

Review the research and ONLY suggest additional searches if there are MAJOR gaps like:
- Completely missing recent developments (last 6 months)
- No statistics or data at all
- Missing a crucial perspective that was specifically asked for

If the research covers the basics reasonably well, say: "The research is sufficient to proceed with the report."

Only suggest 1-2 additional searches if absolutely necessary. We prioritize getting a good report done rather than perfect coverage.""",
)

# 어떻게 사용자가 data나 의견 같은 input을 넣을건지 정의해주는 함수 userproxyagent
# 최종적으로 사용자에게 추가할 상세사항이나 추가 검색 작업 요청 가능
user_proxy = UserProxyAgent(
    "user_proxy",
    description="Human reviewer who can request additional research or approve final results",
    input_func=input,
)

In [6]:
# {roles}는 autogen에 의해 자동으로 교체됨. 모든 후보를 가져와서 저기에 넣어둠
# {history}도 자동으로 대화 내용을 코드 수행하면서 내용 다 넣으면서 리뷰함

selector_prompt = """
Choose the best agent for the current task based on the conversation history:

{roles}

Current conversation:
{history}

Available agents:
- research_planner: Plan the research approach (ONLY at the start)
- research_agent: Search for and extract content from web sources (after planning)
- research_enhancer: Identify CRITICAL gaps only (use sparingly)
- research_analyst: Write the final research report
- quality_reviewer: Check if a complete report exists
- user_proxy: Ask the human for feedback

WORKFLOW:
1. If no planning done yet → select research_planner
2. If planning done but no research → select research_agent  
3. After research_agent completes initial searches → select research_enhancer ONCE
4. If enhancer says "sufficient to proceed" → select research_analyst
5. If enhancer suggests critical searches → select research_agent ONCE more then research_analyst
6. If research_analyst said "REPORT_COMPLETE" → select quality_reviewer
7. If quality_reviewer asked for user feedback → select user_proxy

IMPORTANT: After research_agent has searched 2 times maximum, proceed to research_analyst regardless.

Pick the agent that should work next based on this workflow."""

In [7]:
text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(max_messages=50)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst,
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    # 한 agent가 여러 번 말할 수 있게 해주는 것.
    # 이거 하면 엄청 오래 걸림
    # allow_repeated_speaker=True,
    termination_condition=termination_condition,
)

In [8]:
await Console(
    team.run_stream(task="Research about the current real employment situation in korea and related about ai and korea stock")
    )

---------- TextMessage (user) ----------
Research about the current real employment situation in korea and related about ai and korea stock
---------- TextMessage (research_planner) ----------
### Research Plan: Current Employment Situation in Korea with a Focus on AI and Stock Markets

#### Research Question:
What is the current real employment situation in South Korea, and how is it influenced by developments in artificial intelligence (AI) and its implications for the stock market?

---

### 1. Core Topics:
   - **Employment Trends in South Korea**: Analyze recent data and reports on the employment situation, including unemployment rates, sectors affected, and demographic shifts.
   - **Impact of Artificial Intelligence on the Job Market**: Investigate how AI is changing the employment landscape in South Korea, including job creation and job displacement.
   - **Stock Market Responses to Employment Data and AI Developments**: Examine the correlation between employment trends, AI adv

TaskResult(messages=[TextMessage(id='4a98f5d0-2cc2-4398-92ac-05a18319e72d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 1, 13, 56, 39, 30211, tzinfo=datetime.timezone.utc), content='Research about the current real employment situation in korea and related about ai and korea stock', type='TextMessage'), TextMessage(id='5da39eb0-de60-4099-8a87-13507473d149', source='research_planner', models_usage=RequestUsage(prompt_tokens=129, completion_tokens=436), metadata={}, created_at=datetime.datetime(2026, 3, 1, 13, 56, 50, 513559, tzinfo=datetime.timezone.utc), content='### Research Plan: Current Employment Situation in Korea with a Focus on AI and Stock Markets\n\n#### Research Question:\nWhat is the current real employment situation in South Korea, and how is it influenced by developments in artificial intelligence (AI) and its implications for the stock market?\n\n---\n\n### 1. Core Topics:\n   - **Employment Trends in South Korea**: Analyze recent d